# SNP Sampler Comparison

This notebook converts `examples/demonstrate_snp_sampler.py` into a notebook and compares sampling via inverse CDF root-finding versus accept-reject sampling.

In [ ]:
import os
import sys
import matplotlib.pyplot as plt
import numpy as np
import time

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

from snpdenspy.snp.snp_distribution import SNPDistribution
from snpdenspy.util.diagnose.equal_distribution_plots import (
    qqplot_with_bootstrap_bands,
    ecdfs_with_bands
)

In [ ]:
# --- Create Toy distribution ---
snp_dist = SNPDistribution(K=5)

# Example coefficients to create a bimodal shape
c_arr = [0.5, -0.8, 0.2, 0.0, 0.0]
loc, scale = 12, 5
frozen_dist = snp_dist(c=c_arr, loc=loc, scale=scale)

# Simulate random variates and plot histogram ---
nsamples = 5000

start_time = time.time()
samples_inv_cdf = frozen_dist.rvs(size=nsamples, method="icdf")
end_time = time.time()
elapsed = (end_time - start_time) / 60
print(f"Time taken to generate {nsamples} samples via root-finding: {elapsed:.4f} minutes")

start_time = time.time()
samples_accept_reject = frozen_dist.rvs(size=nsamples, method="ar")
end_time = time.time()
elapsed = (end_time - start_time) / 60
print(f"Time taken to generate {nsamples} samples via accept-reject: {elapsed:.4f} minutes")

# Evaluate PDF on a grid and plot histogram ---
xs = np.linspace(-28, 40, 500)
pdf_vals = frozen_dist.pdf(xs)

plt.figure(figsize=(8,4))
plt.hist(samples_inv_cdf, bins=50, density=True, alpha=0.6, color="orange", label="Ppf")
plt.hist(samples_accept_reject, bins=50, density=True, alpha=0.6, color="red", label="Accept-reject")
plt.plot(xs, pdf_vals, label="PDF", color="blue", lw=2)
plt.title("SNPDistribution (5 coefficients) - Random variates vs PDF")
plt.xlabel("x")
plt.ylabel("Density")
plt.legend()
plt.show()

In [ ]:
# QQ: CI-style bands (separate resampling)
fig1 = qqplot_with_bootstrap_bands(
    samples_inv_cdf, samples_accept_reject,
    n_boot=1000, 
    ci=0.95, 
    bootstrap_mode="ci", 
    random_state=42,
    )[0]

# ECDFs: bootstrap; bootstrap under H0 (pooled) so the Delta band is centered near 0
fig2 = ecdfs_with_bands(
    samples_inv_cdf, samples_accept_reject,
    labels=("Samples with root-finder", "Samples via Accept-reject"),
    bands="bootstrap",
    n_boot=1000,
    ci=0.95,
    bootstrap_mode="ci",  # "ci" or "null"
    random_state=43,
    grid_mode="quantile",
    plot_diff=True,
    title="ECDFs + bands (bootstrap type: pooled)",
    )[0]

plt.show()